# 04 — Machine-learning models

**Question:** does GPS ground deformation improve weekly forecasts beyond what past seismicity already gives?

Phase 5 (`03_baselines`) found that the paper's uplift-driven model tracks the long-term level of seismicity, while a 4-week rolling mean follows swarms. Here we combine both kinds of information in one model and, crucially, compare the **same model with and without GPS features** (an ablation): the difference is what GPS adds.

**Plan:**
1. Features, and a check that none of them uses future data. *(this step)*
2. Poisson regression (GLM), with and without GPS.
3. LightGBM (gradient-boosted trees), with and without GPS.
4. P(M ≥ 3 next week): from the count models, and as direct classifiers.
5. SHAP explanations of the best model.

Same evaluation as phase 5: test weeks January 2020 – December 2025 (312 weeks, plus 2023–2025 separately), walk-forward with expanding window. Training starts in 2013 (before that, uplift is mostly GPS noise and nearly every week is empty). **The bar:** Poisson deviance 14.8 (2020–2025) / 24.1 (2023–2025), Brier 0.107, log-loss 0.335.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.features import (ALL_FEATURES, COMBINED_FEATURES, GPS_FEATURES, SEISMIC_FEATURES,
                          build_features, check_no_lookahead, station_scales)

events = pd.read_parquet("../data/processed/catalogue.parquet")
gps = pd.read_parquet("../data/processed/gps_daily.parquet")
table = build_features(events, gps, start="2005-01-03", end="2025-12-28")
TRAIN_START, TEST_START, TEST_END = "2013-01-07", "2020-01-06", "2025-12-22"
print(f"{len(table)} weeks, {len(ALL_FEATURES)} features")

1095 weeks, 19 features


## 1. Features

All features for the week starting on Monday *w* use data up to the Sunday before (`src/features.py`):

| Group | Feature | Meaning |
|---|---|---|
| Seismicity | `log_count_1w` | log(1 + events with M ≥ 0.6 last week) |
| | `log_mean_2w`, `log_mean_4w`, `log_mean_12w` | log(1 + mean weekly count over the last 2, 4, 12 weeks) |
| | `log_max_4w` | log(1 + largest weekly count in the last 4 weeks): catches swarms |
| | `log_count_m2_4w`, `log_count_m25_4w` | log(1 + events with M ≥ 2, M ≥ 2.5 in the last 4 weeks) |
| | `weeks_since_large` | weeks since the last week with an M ≥ 3 (capped at 260) |
| | `log_N` | log(1 + cumulative count since 2005) |
| GPS | `delta_rite` | RITE uplift since Jan 2005 (cm), 7-day mean |
| | `rate_rite_7d` … `rate_rite_90d` | RITE uplift rate over the last 7, 14, 28, 90 days (cm/year) |
| | `accel_rite` | change of the 28-day rate over 4 weeks |
| | `rate_near_28d` | shared 28-day rate of the 7 near stations (RITE, SOLO, PIS1, STRZ, ARFE, IPPO, ASTR), in RITE units |
| | `rate_far_28d` | mean 28-day rate of 4 far stations (LICO, FRUL, NAMM, MAFE): mostly common noise |
| Combined | `log_N_x_uplift` | log(1 + N × uplift per week): the paper's "rate ∝ N × uplift rate" |
| | `log_paper_mu` | log(1 + the paper's weekly forecast μ, from `03_baselines` section 4) |

**The near-station rate.** Station *i* moves about *sᵢ* times as much as RITE (one source, one shape). Its rate is then about *sᵢ·S*, where *S* is the shared rate. The least-squares estimate from the stations reporting that day is S = Σ sᵢ·rateᵢ / Σ sᵢ². Stations that move little get little weight, so their noise isn't amplified. The scales *sᵢ* are fixed constants, estimated from 2017–2019 (before the test period):

In [2]:
station_scales(gps).round(3).to_frame().T

,RITE,SOLO,PIS1,STRZ,ARFE,IPPO,ASTR
scale,1.0,0.69,0.559,0.571,0.442,0.159,0.231


**Checking for look-ahead.** The most common way to fool yourself in forecasting is a feature that accidentally uses the week being forecast (or later). `check_no_lookahead` deletes all events and GPS data from week *w* onwards, rebuilds the features, and compares week *w*'s row with the full-data version. Any feature that peeks would change.

In [3]:
check_weeks = ["2013-01-07", "2016-06-06", "2020-01-06", "2023-09-25", "2024-05-20", "2025-02-17", "2025-12-22"]
check_no_lookahead(events, gps, "2005-01-03", "2025-12-28", check_weeks, full=table)

,max_abs_difference,features_compared
2013-01-07,0.0,19.0
2016-06-06,0.0,19.0
2020-01-06,0.0,19.0
2023-09-25,0.0,19.0
2024-05-20,0.0,19.0
2025-02-17,0.0,19.0
2025-12-22,0.0,19.0


Every feature is identical: none uses data from its own week or later. (19 of the 19 features are compared; the station scales are the fixed 2017–2019 constants in both cases.)

To show that the check would catch a mistake, here is a deliberately *leaky* feature, the week's own event count, computed both ways:

In [4]:
from src.series import event_counts

def leaky_feature(evts, week):
    """WRONG on purpose: log(1 + this week's own count)."""
    counts = event_counts(evts, "W", mc=0.6, start="2005-01-03", end="2025-12-28")
    return np.log1p(counts.get(pd.Timestamp(week, tz="UTC"), 0))

for week in ["2024-05-20", "2025-02-17"]:
    cut = pd.Timestamp(week, tz="UTC")
    print(f"{week}: full data {leaky_feature(events, week):.2f}, "
          f"data cut at the week {leaky_feature(events[events['time'] < cut], week):.2f}  -> caught")

2024-05-20: full data 5.08, data cut at the week 0.00  -> caught
2025-02-17: full data 5.81, data cut at the week 0.00  -> caught


**How do the features relate to next week's count?** Rank (Spearman) correlation of each feature with log(1 + count) over the training-and-test span 2013–2025. This is only a first look: correlations include the shared upward trend, and models will combine features.

In [5]:
span = table.loc[TRAIN_START:TEST_END]
target = np.log1p(span["count"])
corr = span[ALL_FEATURES].corrwith(target, method="spearman").rename("spearman_with_log_count")
group = pd.Series({f: g for g, fs in [("seismicity", SEISMIC_FEATURES), ("GPS", GPS_FEATURES),
                                      ("combined", COMBINED_FEATURES)] for f in fs}, name="group")
pd.concat([group, corr.round(2)], axis=1).sort_values("spearman_with_log_count", ascending=False)

,group,spearman_with_log_count
delta_rite,GPS,0.79
log_N,seismicity,0.78
log_mean_4w,seismicity,0.77
log_mean_12w,seismicity,0.76
log_max_4w,seismicity,0.75
log_mean_2w,seismicity,0.75
log_N_x_uplift,combined,0.74
log_count_1w,seismicity,0.73
log_paper_mu,combined,0.73
rate_rite_90d,GPS,0.70


In [6]:
# How similar are the GPS features to each other? (Spearman, 2013-2025)
span[GPS_FEATURES].corr(method="spearman").round(2)

,delta_rite,rate_rite_7d,rate_rite_14d,rate_rite_28d,rate_rite_90d,accel_rite,rate_near_28d,rate_far_28d
delta_rite,1.00,0.18,0.37,0.59,0.76,0.03,0.52,0.03
rate_rite_7d,0.18,1.00,0.41,0.22,0.16,0.12,0.23,0.12
rate_rite_14d,0.37,0.41,1.00,0.58,0.39,0.32,0.57,0.28
rate_rite_28d,0.59,0.22,0.58,1.00,0.69,0.51,0.97,0.46
rate_rite_90d,0.76,0.16,0.39,0.69,1.00,-0.07,0.61,0.01
accel_rite,0.03,0.12,0.32,0.51,-0.07,1.00,0.56,0.54
rate_near_28d,0.52,0.23,0.57,0.97,0.61,0.56,1.00,0.59
rate_far_28d,0.03,0.12,0.28,0.46,0.01,0.54,0.59,1.00


**What the features show:**

- **No look-ahead:** all 19 features are identical when every bit of data from the forecast week onwards is deleted, at 7 weeks from 2013 to the last test week; the deliberately leaky feature is caught at once.
- **Slow features correlate most** with next week's count: uplift level (`delta_rite`, 0.79), cumulative count (`log_N`, 0.78), and recent mean counts (0.73–0.77). Much of this is the shared upward trend from 2013 to 2025, so these numbers don't say what helps *week to week*.
- **Short-window uplift rates carry less:** 0.56 for the 28-day rate, 0.36 for 14 days, 0.20 for 7 days. Shorter windows are noisier.
- **Some GPS features are mostly noise or duplicates:**
  - The near-station rate is almost the same as RITE's 28-day rate (correlation 0.97), so averaging stations adds little beyond RITE (as phase 4 suggested).
  - The far-station rate correlates 0.46 with RITE's 28-day rate although the far stations barely move: that's GPS noise common to the whole network (for example seasonal or processing effects). And the acceleration correlates 0.54 with the far-station rate, so it is largely that common noise too.
  - This suggests a cleaner signal for later: RITE's rate *minus* the far-station rate. We'll see first whether the models find the GPS features useful as they are.
- `weeks_since_large` correlates −0.74: long gaps since the last M ≥ 3 mean quiet periods (and the early years, when there were none at all).